# OPAN 6602: Taxi Cancellation Case

**Names:** Pedro Sanchez and Anton van der Laar

This notebook predicts which car service bookings will be cancelled, using a K-Nearest Neighbors classifier. It starts from the raw `Taxi-cancellation-case.csv` file and runs top to bottom. Each question is answered in the markdown cell right after the code that produces it.

In [1]:
### Step 0: Setup ----

# Uncomment and adjust to install libraries not available on Google Colab
# !pip install scikit-learn statsmodels

# Load any libraries used for this project
import pandas as pd
import numpy as np

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.impute import KNNImputer
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, roc_auc_score

In [2]:
### Step 1: Pre-processing ----

# specific Google Colab code to load csv data
from google.colab import files  # so we can upload a CSV to our notebook

uploaded = files.upload()  # opens a file chooser to select our csv file

In [3]:
df = pd.read_csv("Taxi-cancellation-case.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   row#                 10000 non-null  int64  
 1   user_id              10000 non-null  int64  
 2   vehicle_model_id     10000 non-null  int64  
 3   package_id           1752 non-null   float64
 4   travel_type_id       10000 non-null  int64  
 5   from_area_id         9985 non-null   float64
 6   to_area_id           7909 non-null   float64
 7   from_city_id         3706 non-null   float64
 8   to_city_id           339 non-null    float64
 9   from_date            10000 non-null  str    
 10  to_date              5822 non-null   str    
 11  online_booking       10000 non-null  int64  
 12  mobile_site_booking  10000 non-null  int64  
 13  booking_created      10000 non-null  str    
 14  from_lat             9985 non-null   float64
 15  from_long            9985 non-null   float64
 16

In [4]:
# The three timestamps arrive as text. Turn them into real dates so we can do
# arithmetic on them later.
for col in ['from_date', 'to_date', 'booking_created']:
    df[col] = pd.to_datetime(df[col], format='%m/%d/%Y %H:%M')

# row# is just the line number in the file, so it goes now.
df = df.drop(columns=['row#'])

print(len(df), "bookings,", df.shape[1], "columns")

10000 bookings, 18 columns


In [5]:
### Step 2: Split data into train and test sets ----

# Split before anything else. Every step from here until the final evaluation
# looks at the training set only.
#
# Cancellations are a small share of bookings, so stratify keeps the same
# cancellation rate in both halves instead of leaving it to chance.
train, test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['Car_Cancellation']
)

y_train = train['Car_Cancellation']
y_test = test['Car_Cancellation']

print(train.shape, test.shape)
print()
print("Cancellation rate, training set:", round(y_train.mean(), 4))
print("Cancellation rate, test set:    ", round(y_test.mean(), 4))
print()
print("Majority-class baseline, training set:", round(1 - y_train.mean(), 4))
print("Majority-class baseline, test set:    ", round(1 - y_test.mean(), 4))

(8000, 18) (2000, 18)

Cancellation rate, training set: 0.0742
Cancellation rate, test set:     0.0745

Majority-class baseline, training set: 0.9258
Majority-class baseline, test set:     0.9255


**Question 1.**

We split the 10,000 bookings 80/20 at random into a training set (8,000) and a test set (2,000), stratified on `Car_Cancellation` with `random_state=42`.

* **Training set cancellation rate:** 7.42%, so the majority-class baseline is 92.58%.
* **Test set cancellation rate:** 7.45%, so the majority-class baseline is 92.55%.

The two rates are almost identical, which is what stratifying is for. These are the numbers the models have to beat. A model that says "not cancelled" to every booking is already right about 92.6% of the time without learning anything. Everything from here until Question 8 uses the training set only.

In [6]:
### Step 3: Exploratory data analysis ----

# Where are the gaps, in the training data?
miss = train.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]

pd.DataFrame({'n_missing': train[miss.index].isna().sum(),
              'pct_missing': (100 * miss).round(1)})

,n_missing,pct_missing
to_city_id,7735,96.7
package_id,6611,82.6
from_city_id,5021,62.8
to_date,3346,41.8
to_area_id,1654,20.7
to_long,1654,20.7
to_lat,1654,20.7
from_long,13,0.2
from_lat,13,0.2
from_area_id,13,0.2


In [7]:
# Several columns are missing for a big share of rows. Before calling those
# gaps, check whether they line up with the kind of trip. A long distance trip,
# a point to point ride and an hourly rental don't need the same fields.
#
# Each cell is the share of bookings of that trip type missing the column.
# 1 = outstation, 2 = point to point, 3 = hourly rental.
(train[miss.index].isna()
      .groupby(train['travel_type_id'])
      .mean()
      .round(3)
      .T)

travel_type_id,1,2,3
to_city_id,0.000,1.000,1.000
package_id,1.000,1.000,0.000
from_city_id,0.151,0.578,0.945
to_date,0.045,0.446,0.364
to_area_id,1.000,0.000,1.000
to_long,1.000,0.000,1.000
to_lat,1.000,0.000,1.000
from_long,0.049,0.000,0.000
from_lat,0.049,0.000,0.000
from_area_id,0.049,0.000,0.000


In [8]:
# The columns missing for every trip type need a second look.
#
# from_city_id: when it is recorded, is it telling us anything?
print("from_city_id values when present:")
print(train['from_city_id'].value_counts())
print()

# to_date: the recorded values should come after the pickup time.
trip_hours = (train['to_date'] - train['from_date']).dt.total_seconds() / 3600
print("to_date recorded:", train['to_date'].notna().sum())
print("  of which before the pickup time:", (trip_hours < 0).sum())
print("  of which in 1970 (a blank stored as a date):", (train['to_date'].dt.year == 1970).sum())

from_city_id values when present:
from_city_id
15.0    2956
1.0       23
Name: count, dtype: int64

to_date recorded: 4654
  of which before the pickup time: 269
  of which in 1970 (a blank stored as a date): 2


**Question 2.**

Ten columns have gaps in the training set:

| Column | % missing | Verdict |
|---|---|---|
| `to_city_id` | 96.7% | Structural |
| `package_id` | 82.6% | Structural |
| `from_city_id` | 62.8% | Real gap |
| `to_date` | 41.8% | Real gap |
| `to_area_id`, `to_lat`, `to_long` | 20.7% each | Structural |
| `from_area_id`, `from_lat`, `from_long` | 0.2% each (13 rows) | Real gap |

Most of the big gaps line up perfectly with `travel_type_id`. The second table shows each one is either 100% missing or 0% missing depending on the trip type:

* **`package_id`** only exists for hourly rentals. That makes sense, since the package is the hourly deal itself.
* **`to_area_id`, `to_lat`, `to_long`** only exist for point to point rides. Hourly rentals and outstation trips don't have one fixed drop-off spot.
* **`to_city_id`** only exists for outstation trips, the only kind that leaves the city.

So those aren't gaps. The field just doesn't apply, and filling them in would mean inventing a drop-off point for a trip that doesn't have one.

The rest look like real gaps:

* **`to_date`** is missing for 42% of bookings and for every trip type (4.5%, 44.6% and 36.4%). It's also unreliable when it's there: 269 of the recorded values fall before the pickup time, and 2 are dated 1970, which is what a blank looks like after it's been stored as a date.
* **`from_city_id`** is missing across all three trip types too. When it is recorded, it's city 15 in 2,956 of 2,979 cases, so it barely varies.
* **`from_area_id`, `from_lat`, `from_long`** are missing together for 13 bookings, all outstation. The other 95% of outstation trips have a pickup point, so this is a small real gap rather than a rule.

In [9]:
# to_date is the genuine gap to test. It's missing for 4 in 10 bookings across
# every trip type, so it isn't explained by the kind of trip.
#
# Turn "is to_date missing?" into a column we can predict. 1 = missing,
# 0 = present. That makes this a classification problem.
y_miss = train['to_date'].isna().astype(int)

# Predictors: things we know about a booking the moment it comes in.
#
# travel_type_id is a code, not a quantity, so it becomes two 0/1 indicators
# (outstation is the one left out). Same idea for the vehicle model: model 12
# is nearly three quarters of all bookings, so a single indicator for it.
X_miss = pd.DataFrame({
    'point_to_point':      (train['travel_type_id'] == 2).astype(int),
    'hourly':              (train['travel_type_id'] == 3).astype(int),
    'online_booking':      train['online_booking'],
    'mobile_site_booking': train['mobile_site_booking'],
    'common_model':        (train['vehicle_model_id'] == 12).astype(int),
    'pickup_hour':         train['from_date'].dt.hour,
    'booking_day_of_year': train['booking_created'].dt.dayofyear,
})

# KNN needs every feature present. These all are.
print(X_miss.notna().all(axis=1).sum(), "of", len(X_miss), "bookings usable")

8000 of 8000 bookings usable


In [10]:
# The test set stays untouched, so carve a hold-out slice out of the training
# set to grade this diagnostic classifier.
Xm_fit, Xm_hold, ym_fit, ym_hold = train_test_split(
    X_miss, y_miss, test_size=0.2, random_state=42, stratify=y_miss
)

print(Xm_fit.shape, Xm_hold.shape)
print("Share missing to_date:", round(ym_fit.mean(), 3))

(6400, 7) (1600, 7)
Share missing to_date: 0.418


In [11]:
# KNN has one dial: how many neighbors vote. Sweep it and let 10-fold
# cross-validation on the fitting portion say which value generalizes best.
# Scaling sits inside the pipeline so each fold gets its own scaler.
folds = KFold(n_splits=10, shuffle=True, random_state=42)

for k in [1, 3, 5, 7, 11, 15, 21, 31]:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    scores = cross_val_score(pipe, Xm_fit, ym_fit, cv=folds, scoring='accuracy')
    print("k =", k, " CV accuracy:", round(scores.mean(), 3))

k = 1  CV accuracy: 0.957


k = 3  CV accuracy: 0.966


k = 5  CV accuracy: 0.969


k = 7  CV accuracy: 0.968


k = 11  CV accuracy: 0.968


k = 15  CV accuracy: 0.967


k = 21  CV accuracy: 0.967


k = 31  CV accuracy: 0.965


In [12]:
# The curve rises out of k = 1 and flattens by k = 5. Take a value in the
# flat stretch.
knn_miss = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=11))
knn_miss.fit(Xm_fit, ym_fit)

pred_miss = knn_miss.predict(Xm_hold)

# Rows are the truth, columns are the prediction.
print(confusion_matrix(ym_hold, pred_miss))
print()

# The majority class on the hold-out slice: always guess the bigger group.
baseline = max(ym_hold.mean(), 1 - ym_hold.mean())
print("Majority-class baseline:", round(baseline, 3))
print("Accuracy:               ", round(accuracy_score(ym_hold, pred_miss), 3))
print("Precision:              ", round(precision_score(ym_hold, pred_miss), 3))
print("Recall:                 ", round(recall_score(ym_hold, pred_miss), 3))

[[909  22]
 [ 17 652]]

Majority-class baseline: 0.582
Accuracy:                0.976
Precision:               0.967
Recall:                  0.975


In [13]:
# Which feature is doing the work? Drop the booking date and refit.
no_date = ['point_to_point', 'hourly', 'online_booking', 'mobile_site_booking',
           'common_model', 'pickup_hour']

knn_nodate = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=11))
knn_nodate.fit(Xm_fit[no_date], ym_fit)
print("Accuracy without the booking date:",
      round(accuracy_score(ym_hold, knn_nodate.predict(Xm_hold[no_date])), 3))
print()

# And look at it directly: share of bookings missing to_date, by pickup month.
train['to_date'].isna().groupby(train['from_date'].dt.month).mean().round(3)

Accuracy without the booking date: 0.58



from_date
1     0.906
2     0.898
3     0.899
4     0.889
5     0.891
6     0.813
7     0.019
8     0.002
9     0.004
10    0.003
11    0.000
12    0.000
Name: to_date, dtype: float64

**Question 3.**

We tested **`to_date`**, the biggest real gap that affects every trip type.

The outcome is a 0/1 indicator for "`to_date` is missing". The predictors are things known the moment a booking arrives: trip type, booking channel, whether it's the common vehicle model, the pickup hour, and the day of the year the booking was made. We held out 20% of the training set (1,600 bookings) to grade the classifier and ran 10-fold CV on the other 6,400 to pick K. Accuracy climbs from 0.957 at K = 1 and flattens out from K = 5 onward, so we used K = 11.

On the hold-out slice:

|  | Predicted present | Predicted missing |
|---|---|---|
| **Actually present** | 909 | 22 |
| **Actually missing** | 17 | 652 |

* **Accuracy:** 97.6%, against a majority-class baseline of 58.2% on the same slice.
* **Precision:** 96.7%. **Recall:** 97.5%.

That's a huge margin, so the gap is **not Missing Completely at Random**. Taking the booking date out of the model drops accuracy to 58.0%, right back to the baseline. The date is doing all the work. The monthly table shows why: `to_date` is missing for about 90% of bookings from January through May and almost never from July on. It looks like the company started recording return times partway through the year.

What that means: the missingness depends on *when* a booking was made, not on anything about the trip. Dropping those rows would throw out most of the first half of the year, and cancellation rates in those months were very different from later ones. Filling the gap with an average would give January bookings return times learned only from autumn bookings. Neither is safe.

In [14]:
### Step 4: Feature engineering ----

# Decisions for every column with gaps (the reasoning is in the answer below):
#
#   package_id                   structural. Encode as package hours, 0 = no package.
#   to_area_id, to_city_id       structural, and ID codes. Drop.
#   to_lat, to_long              structural. Kept only to build trip features next.
#   from_city_id                 real gap, but one city when present. Drop.
#   to_date                      real gap, not random, often wrong. Drop.
#   from_area_id                 an ID code. Drop.
#   from_lat, from_long          real gap, tiny. KNN-impute.
#
# The package codes stand for hours of hire. Turning the code into hours gives
# the column an order (a 12 hour package really is bigger than a 4 hour one).
package_hours = {1: 4, 2: 8, 3: 6, 4: 10, 5: 5, 6: 3, 7: 12}

def base_table(d):
    return pd.DataFrame({
        'point_to_point':      (d['travel_type_id'] == 2).astype(int),
        'hourly':              (d['travel_type_id'] == 3).astype(int),
        'online_booking':      d['online_booking'],
        'mobile_site_booking': d['mobile_site_booking'],
        'package_hours':       d['package_id'].map(package_hours).fillna(0),
        'from_lat':            d['from_lat'],
        'from_long':           d['from_long'],
    }, index=d.index)

X_train = base_table(train)
X_test = base_table(test)

X_train.isna().sum()

point_to_point          0
hourly                  0
online_booking          0
mobile_site_booking     0
package_hours           0
from_lat               13
from_long              13
dtype: int64

In [15]:
# KNNImputer fills a gap with the average of the k most similar bookings that
# do report it. That's a KNN regression, so run one ourselves first to pick k.
#
# Predict pickup latitude from the other booking fields, using only the
# training bookings that report it.
others = ['point_to_point', 'hourly', 'online_booking',
          'mobile_site_booking', 'package_hours']

has_pickup = X_train['from_lat'].notna()
X_lat = X_train.loc[has_pickup, others]
y_lat = X_train.loc[has_pickup, 'from_lat']

print(len(X_lat), "bookings report a pickup point")
print("Pickup latitude: std", round(y_lat.std(), 4))
print()

# scikit-learn reports the negative of RMSE. Flip the sign.
folds = KFold(n_splits=10, shuffle=True, random_state=42)

for k in [1, 3, 5, 11, 21, 31, 51, 101, 201]:
    pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    scores = cross_val_score(pipe, X_lat, y_lat, cv=folds,
                             scoring='neg_root_mean_squared_error')
    print("k =", k, " CV RMSE:", round(-scores.mean(), 4))

7987 bookings report a pickup point
Pickup latitude: std 0.0862



k = 1  CV RMSE: 0.1434


k = 3  CV RMSE: 0.1129


k = 5  CV RMSE: 0.1012


k = 11  CV RMSE: 0.0939


k = 21  CV RMSE: 0.0894


k = 31  CV RMSE: 0.087


k = 51  CV RMSE: 0.0858


k = 101  CV RMSE: 0.0858


k = 201  CV RMSE: 0.0855


In [16]:
# The error falls until about k = 51 and then goes flat. Use that.
#
# Scale first, impute second, so a column measured in hours doesn't outvote
# one measured in degrees. Both are fit on the training set only. Calling
# transform on the test set reuses the training scaler, and KNNImputer only
# looks for neighbors among the training rows it was fit on, so a test
# booking is never filled in from another test booking.
prep = make_pipeline(StandardScaler(), KNNImputer(n_neighbors=51))
prep.fit(X_train)

scaler = prep.named_steps['standardscaler']

filled_train = pd.DataFrame(scaler.inverse_transform(prep.transform(X_train)),
                            columns=X_train.columns, index=X_train.index)
filled_test = pd.DataFrame(scaler.inverse_transform(prep.transform(X_test)),
                           columns=X_test.columns, index=X_test.index)

print("Gaps remaining:", int(filled_train.isna().sum().sum()), "train,",
      int(filled_test.isna().sum().sum()), "test")

Gaps remaining: 0 train, 0 test


In [17]:
# Sanity check 1: the column means should barely move.
cols = ['from_lat', 'from_long']
pd.DataFrame({
    'observed_mean': X_train[cols].mean().round(4),
    'after_imputation': filled_train[cols].mean().round(4),
    'n_missing': X_train[cols].isna().sum(),
})

,observed_mean,after_imputation,n_missing
from_lat,12.9840,12.9840,13
from_long,77.6363,77.6364,13


In [18]:
# Sanity check 2: the filled values sit inside the range of real pickups.
imputed = filled_train.loc[~has_pickup, cols]

print("Imputed latitude: ", round(imputed['from_lat'].min(), 4), "to", round(imputed['from_lat'].max(), 4))
print("Observed latitude:", round(X_train['from_lat'].min(), 4), "to", round(X_train['from_lat'].max(), 4))
print("Imputed longitude: ", round(imputed['from_long'].min(), 4), "to", round(imputed['from_long'].max(), 4))
print("Observed longitude:", round(X_train['from_long'].min(), 4), "to", round(X_train['from_long'].max(), 4))

Imputed latitude:  12.9903 to 13.0194
Observed latitude: 12.7766 to 13.3661
Imputed longitude:  77.6423 to 77.6491
Observed longitude: 77.3869 to 77.7864


**Question 4.**

Here's what we did with each column that has gaps, and why:

* **`package_id` → encoded.** The gap is structural (Question 2). We turned the package code into its number of hours (4, 8, 6 and so on), with 0 meaning "not an hourly rental". Hours have a real order, and the raw codes don't.
* **`to_area_id`, `to_city_id` → dropped.** Both are structural, and both are ID codes with hundreds of values that a distance-based model can't use. The trip type indicators already say whether a booking has a drop-off area or a destination city.
* **`to_lat`, `to_long` → kept for feature engineering, not imputed.** They're structural, so there's nothing to fill. Question 5 uses them to build trip features, then the raw columns leave the table.
* **`from_city_id` → dropped.** A real gap, but when it's present it's the same city 99% of the time. A column that barely changes can't help separate bookings.
* **`to_date` → dropped.** Question 3 showed the gap depends on when the booking was made, and a lot of the recorded values are wrong anyway. We also thought about a "missing" flag, but that flag would really just mean "booked before July". Every live booking would score 0 on it, so the model would be learning something that won't exist when it's used.
* **`from_area_id` → dropped.** It's an ID code, and the pickup coordinates carry the same information in a form KNN can use.
* **`from_lat`, `from_long` → KNN-imputed.** A small real gap (13 training bookings). Dropping those rows would drop a slice of outstation trips, and they're already the rarest trip type.

For the imputation, we scaled the features first and picked K by 10-fold CV on the 7,987 training bookings that do report a pickup point, predicting latitude from the other booking fields. RMSE falls from 0.143 at K = 1 to 0.086 at K = 51 and stays flat after that, so we used **K = 51**. The scaler and imputer were fit on the training set only, then applied unchanged to the test set. KNNImputer only borrows neighbors from the rows it was fit on, so no test booking got filled in from another test booking.

Sanity checks passed. The mean latitude didn't move at all (12.9840 before and after), and the mean longitude moved by 0.0001. The imputed points all sit well inside the real range of pickups. One thing worth saying honestly: the best RMSE (0.086) is about the same as the plain spread of pickup latitudes (0.086). The other booking fields don't really know where a pickup happens, so the imputed points land close to the typical outstation pickup. With 13 rows out of 8,000 that's harmless, but the imputation isn't adding much information.

In [19]:
# New features. Two come from the timestamps, two from the coordinates, one
# from the vehicle code.
#
# The airport: the single most common drop-off point in the training data.
# Learn it from the training set and reuse it for the test set.
top_dropoff = train.groupby(['to_lat', 'to_long']).size().idxmax()
print("Most common drop-off point:", top_dropoff,
      " bookings:", train.groupby(['to_lat', 'to_long']).size().max())

def distance_km(lat1, lon1, lat2, lon2):
    # Straight line distance on the globe (haversine formula)
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 6371 * 2 * np.arcsin(np.sqrt(a))

def add_features(filled, d):
    out = filled.copy()

    # Hours between making the booking and the pickup
    out['lead_hours'] = (d['from_date'] - d['booking_created']).dt.total_seconds() / 3600

    # Pickup time of day, placed on a clock face so 11pm sits next to midnight
    hour = d['from_date'].dt.hour + d['from_date'].dt.minute / 60
    out['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    out['hour_cos'] = np.cos(2 * np.pi * hour / 24)

    # Trip length for point to point rides. Other trip types have no single
    # drop-off point, so they get 0, and point_to_point tells them apart.
    out['trip_km'] = distance_km(out['from_lat'], out['from_long'],
                                 d['to_lat'], d['to_long']).fillna(0)

    # Does the trip start or end at the airport?
    at_airport = lambda la, lo: (la == top_dropoff[0]) & (lo == top_dropoff[1])
    out['airport_trip'] = (at_airport(d['from_lat'], d['from_long'])
                           | at_airport(d['to_lat'], d['to_long'])).astype(int)

    # The vehicle model that makes up most of the fleet
    out['common_model'] = (d['vehicle_model_id'] == 12).astype(int)
    return out

feat_train = add_features(filled_train, train)
feat_test = add_features(filled_test, test)

# A first look: cancellation rate for each new flag
for f in ['airport_trip', 'common_model']:
    print(y_train.groupby(feat_train[f]).mean().round(3).to_dict(), "<-", f)

Most common drop-off point: (np.float64(13.19956), np.float64(77.70688))  bookings: 1632
{0: 0.097, 1: 0.021} <- airport_trip
{0: 0.044, 1: 0.086} <- common_model


In [20]:
# Screen for outliers and impossible values.
feat_train.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
point_to_point,8000.0,0.79,0.40,0.00,1.00,1.00,1.00,1.00
hourly,8000.0,0.17,0.38,0.00,0.00,0.00,0.00,1.00
online_booking,8000.0,0.35,0.48,0.00,0.00,0.00,1.00,1.00
mobile_site_booking,8000.0,0.04,0.20,0.00,0.00,0.00,0.00,1.00
package_hours,8000.0,1.03,2.43,0.00,0.00,0.00,0.00,12.00
from_lat,8000.0,12.98,0.09,12.78,12.93,12.97,13.01,13.37
from_long,8000.0,77.64,0.06,77.39,77.59,77.64,77.69,77.79
lead_hours,8000.0,35.87,98.61,-0.40,3.05,8.78,18.50,1222.17
hour_sin,8000.0,0.02,0.76,-1.00,-0.81,0.00,0.81,1.00
hour_cos,8000.0,-0.14,0.64,-1.00,-0.73,-0.25,0.37,1.00


In [21]:
# A booking made after its own pickup time can't happen.
print("Negative lead time:", (feat_train['lead_hours'] < 0).sum(), "train,",
      (feat_test['lead_hours'] < 0).sum(), "test")

# Point to point rides that start and end at the same spot
p2p = feat_train['point_to_point'] == 1
print("Point to point rides under 100 meters:", (p2p & (feat_train['trip_km'] < 0.1)).sum())

# How long is the tail on lead time?
print()
print(feat_train['lead_hours'].quantile([0.5, 0.9, 0.99, 1]).round(1))

Negative lead time: 4 train, 4 test
Point to point rides under 100 meters: 5

0.50       8.8
0.90      66.5
0.99     597.7
1.00    1222.2
Name: lead_hours, dtype: float64


In [22]:
# Fixes, all learned from the training set and applied to both sets:
#
#   Negative lead time: set to 0, as if booked at the moment of pickup.
#   Lead time tail: cap at the training set's 99th percentile. A handful of
#   bookings made weeks ahead would otherwise stretch the scale for everyone.
lead_cap = feat_train['lead_hours'].quantile(0.99)

for d in (feat_train, feat_test):
    d['lead_hours'] = d['lead_hours'].clip(lower=0, upper=lead_cap)

print("Lead time cap:", round(lead_cap, 1), "hours (about", round(lead_cap / 24), "days)")
print("Train range:", feat_train['lead_hours'].min(), "to", round(feat_train['lead_hours'].max(), 1))
print("Test range: ", feat_test['lead_hours'].min(), "to", round(feat_test['lead_hours'].max(), 1))

Lead time cap: 597.7 hours (about 25 days)
Train range: 0.0 to 597.7
Test range:  0.0 to 597.7


**Question 5.**

We built five new features:

* **`lead_hours`** (from the timestamps): hours between making the booking and the pickup. Someone who books at the last minute, or weeks ahead, has more chances for plans to change.
* **`hour_sin`, `hour_cos`** (from the pickup time): the pickup hour placed on a clock face so 11pm and midnight end up next to each other. Evening and late night rides plausibly carry different risk than a morning commute.
* **`trip_km`** (from the coordinates): straight line distance for point to point rides, 0 for the other trip types, which have no single drop-off. A very short trip is easy to replace with an auto rickshaw or a walk, which could lead to a cancellation.
* **`airport_trip`** (from the coordinates): 1 if the ride starts or ends at the most common drop-off point in the training data (13.19956, 77.70688). That's Bangalore's Kempegowda International Airport. People don't cancel a ride to catch a flight lightly. In training, airport trips cancel 2.1% of the time against 9.7% for everything else.
* **`common_model`** (from the vehicle code): 1 for vehicle model 12, which is 73% of bookings. Model 12 bookings cancel 8.6% of the time against 4.4% for the rest.

**Screening for outliers and impossible values:**

* **Negative lead times.** 4 training bookings (and 4 test bookings) were created *after* their pickup time, which can't happen. We set them to 0, as if booked at the moment of pickup.
* **Very long lead times.** The median booking is made 8.8 hours ahead, but the maximum is 1,222 hours (about 51 days). We capped lead time at the training set's 99th percentile, 597.7 hours (about 25 days), and used that same cutoff on the test set. Without the cap, a handful of far-ahead bookings would stretch the scale and squash everyone else together.
* **Coordinates.** Every latitude and longitude falls between 12.78 and 13.37 N and between 77.39 and 77.79 E, which is the Bangalore area. Nothing to fix.
* **Trip distance.** 0 to 53 km, plausible for the city. 5 point to point rides start and end within 100 meters of each other. That's odd, but 5 rows can't move the model, so we left them alone.
* **Package hours** only take the published package values, so no issues there.

In [23]:
# Redundancy check on the training set. Two features that measure the same
# thing get counted twice in the distance.
corr = feat_train.corr()

pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
             .stack()
             .rename('r'))
pairs[pairs.abs() >= 0.7].sort_values(key=abs, ascending=False).round(2)

hourly          package_hours    0.92
point_to_point  hourly          -0.90
                package_hours   -0.83
trip_km         airport_trip     0.80
Name: r, dtype: float64

In [24]:
# hourly and package_hours: package_hours is above 0 exactly when the trip is
# an hourly rental, so it already carries the hourly flag. Drop hourly.
print("Share of bookings where hourly = 1 exactly when package_hours > 0:",
      ((feat_train['hourly'] == 1) == (feat_train['package_hours'] > 0)).mean())
print("Average trip_km, airport trips vs the rest:",
      feat_train.groupby('airport_trip')['trip_km'].mean().round(1).to_dict())
print()

# trip_km and airport_trip are the other close pair. Keep whichever one the
# model does better with, using 10-fold CV on the training set.
folds_strat = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

all_feats = [c for c in feat_train.columns if c != 'hourly']
options = {
    'both':          all_feats,
    'drop trip_km':  [c for c in all_feats if c != 'trip_km'],
    'drop airport':  [c for c in all_feats if c != 'airport_trip'],
}

for name, cols in options.items():
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=51))
    scores = cross_val_score(pipe, feat_train[cols], y_train, cv=folds_strat,
                             scoring='roc_auc')
    print(name.ljust(14), "CV AUC:", round(scores.mean(), 3))

Share of bookings where hourly = 1 exactly when package_hours > 0: 1.0
Average trip_km, airport trips vs the rest: {0: 7.7, 1: 28.1}



both           CV AUC: 0.78


drop trip_km   CV AUC: 0.782


drop airport   CV AUC: 0.773


In [25]:
# Final predictor set, applied the same way to both sets.
features = [c for c in feat_train.columns if c not in ['hourly', 'trip_km']]

Z_train = feat_train[features]
Z_test = feat_test[features]

print(len(features), "features:", features)

11 features: ['point_to_point', 'online_booking', 'mobile_site_booking', 'package_hours', 'from_lat', 'from_long', 'lead_hours', 'hour_sin', 'hour_cos', 'airport_trip', 'common_model']


**Question 6.**

Three pairs of features have a correlation of 0.8 or more in the training set:

* **`hourly` and `package_hours` (r = 0.92).** `package_hours` is above 0 for every hourly rental and only for hourly rentals, so it already contains the hourly flag plus the size of the package. We **dropped `hourly`**. That also takes care of the second pair, `point_to_point` and `hourly` (r = -0.90), which were nearly mirror images because outstation trips are so rare.
* **`trip_km` and `airport_trip` (r = 0.80).** Airport runs average 28.1 km against 7.7 km for everything else, so a long trip and an airport trip largely describe the same bookings. Keeping both would count that distance twice. We compared the two options with 10-fold CV on the training set at K = 51. Dropping `trip_km` gave a CV AUC of 0.782, dropping `airport_trip` gave 0.773, and keeping both gave 0.780. So we **dropped `trip_km`** and kept the airport flag, which does the same job slightly better.

`point_to_point` and `package_hours` still sit at r = -0.83, but they aren't measuring the same thing. One says whether it's a point to point ride, the other says how big the hourly package is. Between them they're how the model tells all three trip types apart, so both stay.

This check stays with correlations on purpose. VIF is a post-regression diagnostic, so it comes after the logistic regression in Question 9, where it checks that regression's coefficients.

The final model uses **11 features**, and the test set got exactly the same list: `point_to_point`, `online_booking`, `mobile_site_booking`, `package_hours`, `from_lat`, `from_long`, `lead_hours`, `hour_sin`, `hour_cos`, `airport_trip`, `common_model`.

In [26]:
### Step 5: Feature and model selection ----

# Sweep k with 10-fold CV on the training set. Folds are stratified so each
# one gets its fair share of the rare cancellations.
#
# Accuracy on its own won't separate the candidates: the baseline is already
# above 92%. So report AUC too. It measures how well the model ranks bookings,
# which is what we need when we pick a cutoff later.
for k in [1, 3, 5, 11, 21, 31, 51, 75, 101, 151, 201, 301]:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    acc = cross_val_score(pipe, Z_train, y_train, cv=folds_strat, scoring='accuracy')
    auc = cross_val_score(pipe, Z_train, y_train, cv=folds_strat, scoring='roc_auc')
    print("k =", str(k).ljust(4), " CV accuracy:", round(acc.mean(), 3),
          "  CV AUC:", round(auc.mean(), 3))

k = 1     CV accuracy: 0.883   CV AUC: 0.572


k = 3     CV accuracy: 0.913   CV AUC: 0.647


k = 5     CV accuracy: 0.921   CV AUC: 0.685


k = 11    CV accuracy: 0.925   CV AUC: 0.722


k = 21    CV accuracy: 0.926   CV AUC: 0.754


k = 31    CV accuracy: 0.927   CV AUC: 0.77


... (2 outputs left out)

k = 101   CV accuracy: 0.927   CV AUC: 0.788


k = 151   CV accuracy: 0.926   CV AUC: 0.783


k = 201   CV accuracy: 0.926   CV AUC: 0.781


k = 301   CV accuracy: 0.926   CV AUC: 0.774


In [27]:
# AUC climbs until about k = 101 and then slowly slides. Take the peak.
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=101))

# Before fitting on everything, see how different cutoffs behave. These are
# out-of-fold predictions, so every training booking is scored by a model
# that never saw it.
cv_prob = cross_val_predict(knn, Z_train, y_train, cv=folds_strat,
                            method='predict_proba')[:, 1]

rows = []
for t in [0.5, 0.3, 0.2, 0.15, 0.1, 0.075, 0.05]:
    flag = (cv_prob >= t).astype(int)
    rows.append({'cutoff': t,
                 'share_flagged': round(flag.mean(), 3),
                 'precision': round(precision_score(y_train, flag, zero_division=0), 3),
                 'recall': round(recall_score(y_train, flag), 3)})

pd.DataFrame(rows)

,cutoff,share_flagged,precision,recall
0,0.500,0.004,0.583,0.035
1,0.300,0.036,0.387,0.187
2,0.200,0.085,0.281,0.322
3,0.150,0.150,0.235,0.476
4,0.100,0.264,0.182,0.648
5,0.075,0.357,0.159,0.766
6,0.050,0.458,0.136,0.838


In [28]:
knn.fit(Z_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('kneighborsclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",101
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on the

**Question 7.**

We swept K from 1 to 301 with 10-fold CV on the training set (stratified folds, scaling inside the pipeline).

Accuracy on its own can't pick a winner. From K = 21 up, every value lands at 0.926 or 0.927, which is just the 92.6% baseline. The model gets there by calling almost every booking "not cancelled". So we also scored each K on AUC, which measures how well the model *ranks* bookings from risky to safe. That's the part that matters, since the cutoff gets chosen separately.

| K | CV accuracy | CV AUC |
|---|---|---|
| 1 | 0.883 | 0.572 |
| 11 | 0.925 | 0.722 |
| 31 | 0.927 | 0.770 |
| 51 | 0.927 | 0.782 |
| **101** | **0.927** | **0.788** |
| 201 | 0.926 | 0.781 |
| 301 | 0.926 | 0.774 |

AUC climbs steadily out of K = 1, peaks at **K = 101**, then slowly drops as the neighborhood grows big enough to blur real differences. So the final model is KNN with K = 101, fit on all 8,000 training bookings.

The cutoff table uses out-of-fold predictions from the same CV, so it's still training data only. It shows the trade-off Question 8 has to settle. At the default 0.5 cutoff, the model flags 0.4% of bookings and catches 3.5% of cancellations. At 0.10, it flags 26% and catches 65%.

In [29]:
### Step 6: Predictions and final model evaluation ----

# The test set went through the same steps: the training scaler and imputer
# (Step 4), the same engineered features, the training lead time cap, and the
# same final feature list. Nothing was refit on it.
test_prob = knn.predict_proba(Z_test)[:, 1]

def report(cutoff):
    pred = (test_prob >= cutoff).astype(int)
    print("Cutoff:", cutoff)
    print(confusion_matrix(y_test, pred))
    print("Baseline (predict 'not cancelled' always):", round(1 - y_test.mean(), 3))
    print("Accuracy: ", round(accuracy_score(y_test, pred), 3))
    print("Precision:", round(precision_score(y_test, pred, zero_division=0), 3))
    print("Recall:   ", round(recall_score(y_test, pred), 3))
    print("Share of bookings flagged:", round(pred.mean(), 3))
    print()
    return pred

# AUC doesn't depend on the cutoff. Compare it with the CV value for k = 101.
print("Test AUC:", round(roc_auc_score(y_test, test_prob), 3))
print()

pred_default = report(0.5)
pred_chosen = report(0.1)

Test AUC: 0.778

Cutoff: 0.5
[[1842    9]
 [ 146    3]]
Baseline (predict 'not cancelled' always): 0.926
Accuracy:  0.922
Precision: 0.25
Recall:    0.02
Share of bookings flagged: 0.006

Cutoff: 0.1
[[1425  426]
 [  62   87]]
Baseline (predict 'not cancelled' always): 0.926
Accuracy:  0.756
Precision: 0.17
Recall:    0.584
Share of bookings flagged: 0.256



**Question 8.**

The test set went through exactly the same steps as the training set: training scaler and imputer, the same engineered features, the training lead time cap and the same 11 features. Nothing was refit.

**At the default cutoff (0.5):**

|  | Predicted kept | Predicted cancelled |
|---|---|---|
| **Actually kept** | 1,842 | 9 |
| **Actually cancelled** | 146 | 3 |

Accuracy is 92.2%, slightly *below* the 92.6% baseline. Precision is 25% and recall is 2%. It catches 3 of 149 cancellations, which makes it useless in practice.

**At the cutoff we'd recommend (0.10), chosen from the training CV table in Question 7:**

|  | Predicted kept | Predicted cancelled |
|---|---|---|
| **Actually kept** | 1,425 | 426 |
| **Actually cancelled** | 62 | 87 |

Accuracy is 75.6%, below the baseline, and that's expected once the model starts flagging bookings. Precision is 17.0%, recall is 58.4%, and 25.6% of bookings get flagged. Test AUC is 0.778, close to the 0.788 from cross-validation, so the model held up on bookings it had never seen.

**Which matters more:** recall. Think about what each mistake costs the operations team. A false alarm means someone sends a confirmation text or makes a quick call to a customer who was going to ride anyway. A missed cancellation means a driver and car are tied up for a job that disappears, the slot could have gone to someone else, and the driver's time is lost. The second one costs a lot more, so the team should care most about catching cancellations, even if that means checking on some bookings that were fine.

**What that means for the cutoff:** go well below 0.5. A cancellation rate of 7.4% means hardly any booking ever looks more than 50% likely to cancel, so a 0.5 cutoff almost never fires. At 0.10, the model catches close to 6 in 10 cancellations, and a flagged booking is about 2.3 times as likely to cancel as an average one (17.0% against 7.45%). If the team can only handle a smaller list, 0.15 flags about 15% of bookings and still catches around half the cancellations (training CV). The right number depends on how many follow-ups they can realistically do each day.

In [30]:
# Profile the bookings the model flags (cutoff 0.1) against the rest, on the
# test set. Use the raw fields where they are easier to read.
profile = pd.DataFrame({
    'flagged':             pred_chosen,
    'actually_cancelled':  y_test.values,
    'booked_on_website':   test['online_booking'].values,
    'booked_on_mobile':    test['mobile_site_booking'].values,
    'booked_by_phone':     ((test['online_booking'] == 0) & (test['mobile_site_booking'] == 0)).astype(int).values,
    'point_to_point':      (test['travel_type_id'] == 2).astype(int).values,
    'hourly_rental':       (test['travel_type_id'] == 3).astype(int).values,
    'outstation':          (test['travel_type_id'] == 1).astype(int).values,
    'airport_trip':        feat_test['airport_trip'].values,
    'common_model':        feat_test['common_model'].values,
    'evening_pickup':      test['from_date'].dt.hour.between(17, 23).astype(int).values,
    'late_night_pickup':   test['from_date'].dt.hour.between(0, 5).astype(int).values,
    'lead_hours':          feat_test['lead_hours'].values,
    'trip_km':             feat_test['trip_km'].values,
})

summary = profile.groupby('flagged').mean().T.round(2)
summary.columns = ['not flagged', 'flagged']
summary.loc['bookings'] = profile['flagged'].value_counts().sort_index().values
summary

,not flagged,flagged
actually_cancelled,0.04,0.17
booked_on_website,0.28,0.62
booked_on_mobile,0.01,0.14
booked_by_phone,0.71,0.24
point_to_point,0.73,0.93
hourly_rental,0.22,0.06
outstation,0.05,0.01
airport_trip,0.35,0.03
common_model,0.69,0.83
evening_pickup,0.23,0.44


In [31]:
# Averages can hide a long tail, and lead time has one. Break it into bands
# and look at each band: how often it cancels, and how often the model flags it.
bands = pd.cut(profile['lead_hours'], [-1, 3, 24, 168, 10000],
               labels=['under 3 hours', '3 to 24 hours', '1 to 7 days', 'over a week'])

profile.groupby(bands, observed=True).agg(
    bookings=('flagged', 'size'),
    cancel_rate=('actually_cancelled', 'mean'),
    share_flagged=('flagged', 'mean'),
).round(3)

,bookings,cancel_rate,share_flagged
lead_hours,,,
under 3 hours,510,0.073,0.257
3 to 24 hours,1101,0.081,0.268
1 to 7 days,287,0.038,0.150
over a week,102,0.118,0.431


In [32]:
# The profile says which bookings are riskier. It doesn't say by how much once
# everything else is held equal. Website bookings and evening pickups overlap,
# for example, so each one's effect is tangled up with the other's.
#
# KNN can't untangle that, so fit a logistic regression on the training set as
# an explanation tool, not a replacement. Same predictors, written in a form a
# manager can read:
#
#   time of day   evening (5pm to 11pm) and late night (midnight to 5am),
#                 compared with daytime
#   lead time     three bands, compared with booking 1 to 7 days ahead. The
#                 band table above showed the effect isn't a straight line.
#   channel       website and mobile site, compared with a phone booking
def readable(filled, d):
    hour = d['from_date'].dt.hour
    lead = filled['lead_hours']
    return pd.DataFrame({
        'point_to_point':      filled['point_to_point'],
        'online_booking':      filled['online_booking'],
        'mobile_site_booking': filled['mobile_site_booking'],
        'package_hours':       filled['package_hours'],
        'airport_trip':        filled['airport_trip'],
        'common_model':        filled['common_model'],
        'evening_pickup':      hour.between(17, 23).astype(int),
        'late_night_pickup':   hour.between(0, 5).astype(int),
        'booked_under_3h':     (lead < 3).astype(int),
        'booked_3_to_24h':     ((lead >= 3) & (lead < 24)).astype(int),
        'booked_over_week':    (lead >= 168).astype(int),
        'from_lat':            filled['from_lat'],
        'from_long':           filled['from_long'],
    }, index=filled.index)

R_train = readable(feat_train, train)
R_test = readable(feat_test, test)

logit = sm.Logit(y_train, sm.add_constant(R_train)).fit(disp=0)
print(logit.summary())

                           Logit Regression Results                           
Dep. Variable:       Car_Cancellation   No. Observations:                 8000
Model:                          Logit   Df Residuals:                     7986
Method:                           MLE   Df Model:                           13
Date:                Mon, 28 Sep 2026   Pseudo R-squ.:                  0.1586
Time:                        23:21:50   Log-Likelihood:                -1780.4
converged:                       True   LL-Null:                       -2116.0
Covariance Type:            nonrobust   LLR p-value:                5.106e-135
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                -330.2340     63.007     -5.241      0.000    -453.726    -206.742
point_to_point          0.5975      0.245      2.441      0.015       0.118       1.077
online_booking  

In [33]:
# Post-regression check: VIF. Each predictor is regressed on all the others.
# A VIF near 1 means it brings its own information. Above 5 or so, the
# coefficient is being blurred by overlap with the rest.
Xc = sm.add_constant(R_train)
vif = pd.Series([variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])],
                index=R_train.columns, name='VIF')
vif.round(2).sort_values(ascending=False)

point_to_point         3.31
package_hours          3.21
booked_3_to_24h        2.19
booked_under_3h        2.18
airport_trip           1.47
booked_over_week       1.31
late_night_pickup      1.26
from_lat               1.24
evening_pickup         1.17
from_long              1.09
online_booking         1.06
mobile_site_booking    1.04
common_model           1.03
Name: VIF, dtype: float64

In [34]:
# Logit coefficients are in log-odds, which nobody on an operations team reads.
# Average marginal effects turn them into something they can: the change in
# the chance of cancelling, in percentage points, for a booking that is the
# same in every other way. For the 0/1 features this is the jump from 0 to 1.
#
# The coordinates stay in the model as controls. A one-degree move is about
# 110 km, so their effects aren't meaningful on their own.
me = logit.get_margeff(at='overall', dummy=True).summary_frame()
me = me.drop(index=['from_lat', 'from_long'])

effects = pd.DataFrame({
    'effect_pct_points': (100 * me['dy/dx']).round(1),
    'low_95':            (100 * me.iloc[:, 4]).round(1),   # lower end of the 95% interval
    'high_95':           (100 * me.iloc[:, 5]).round(1),   # upper end
    'p_value':           me['Pr(>|z|)'].round(3),
}).sort_values('effect_pct_points')

print("Average cancellation rate, training set:", round(100 * y_train.mean(), 1), "%")
print("Logit test AUC:", round(roc_auc_score(y_test, logit.predict(sm.add_constant(R_test))), 3))
effects

Average cancellation rate, training set: 7.4 %
Logit test AUC: 0.774


,effect_pct_points,low_95,high_95,p_value
airport_trip,-7.3,-8.3,-6.2,0.000
late_night_pickup,-1.7,-3.5,0.1,0.059
package_hours,-0.3,-0.9,0.2,0.235
common_model,2.9,1.7,4.0,0.000
evening_pickup,3.1,1.8,4.4,0.000
point_to_point,3.2,0.9,5.5,0.006
booked_3_to_24h,4.5,2.2,6.8,0.000
booked_under_3h,8.4,5.0,11.9,0.000
online_booking,10.2,8.7,11.6,0.000
booked_over_week,13.0,6.8,19.2,0.000


In [35]:
# Does the KNN model tell the same story? Take every test booking, change one
# thing, and see how the model's average predicted risk moves. It's the KNN
# version of a marginal effect.
def avg_risk(changes):
    Z = Z_test.copy()
    for col, val in changes.items():
        Z[col] = val
    return knn.predict_proba(Z)[:, 1].mean()

def clock(hour):
    return {'hour_sin': np.sin(2 * np.pi * hour / 24), 'hour_cos': np.cos(2 * np.pi * hour / 24)}

phone = avg_risk({'online_booking': 0, 'mobile_site_booking': 0})
whatif = pd.DataFrame([
    ['website instead of phone', phone, avg_risk({'online_booking': 1, 'mobile_site_booking': 0})],
    ['mobile site instead of phone', phone, avg_risk({'online_booking': 0, 'mobile_site_booking': 1})],
    ['airport trip instead of not', avg_risk({'airport_trip': 0}), avg_risk({'airport_trip': 1})],
    ['7pm pickup instead of 11am', avg_risk(clock(11)), avg_risk(clock(19))],
    ['booked 2 hours ahead instead of 3 days', avg_risk({'lead_hours': 72}), avg_risk({'lead_hours': 2})],
    ['booked 10 days ahead instead of 3 days', avg_risk({'lead_hours': 72}), avg_risk({'lead_hours': 240})],
], columns=['change', 'risk_before', 'risk_after'])

whatif['effect_pct_points'] = (100 * (whatif['risk_after'] - whatif['risk_before'])).round(1)
whatif[['risk_before', 'risk_after']] = (100 * whatif[['risk_before', 'risk_after']]).round(1)
whatif

,change,risk_before,risk_after,effect_pct_points
0,website instead of phone,4.2,13.0,8.8
1,mobile site instead of phone,4.2,18.4,14.2
2,airport trip instead of not,10.0,3.0,-7.0
3,7pm pickup instead of 11am,6.0,11.0,5.0
4,booked 2 hours ahead instead of 3 days,7.5,7.9,0.3
5,booked 10 days ahead instead of 3 days,7.5,6.3,-1.3


In [36]:
# Size the two groups the recommendations target, on the training set.
hour = train['from_date'].dt.hour
booked_online = (train['online_booking'] == 1) | (train['mobile_site_booking'] == 1)
city_ride = (train['travel_type_id'] == 2) & (feat_train['airport_trip'] == 0)

groups = {
    'evening online city rides (not airport)': hour.between(17, 23) & booked_online & city_ride,
    'phone bookings to or from the airport':   ~booked_online & (feat_train['airport_trip'] == 1),
}

for name, g in groups.items():
    print(name)
    print("  share of bookings:     ", round(100 * g.mean(), 1), "%")
    print("  cancellation rate:     ", round(100 * y_train[g].mean(), 1), "%")
    print("  share of cancellations:", round(100 * y_train[g].sum() / y_train.sum(), 1), "%")
    print()

# What the 0.10 cutoff means per 1,000 bookings, from the test set
print("Per 1,000 bookings: flagged", round(1000 * pred_chosen.mean()),
      "| cancellations", round(1000 * y_test.mean()),
      "| cancellations inside the flagged list", round(1000 * (pred_chosen * y_test.values).mean()))

evening online city rides (not airport)
  share of bookings:      7.6 %
  cancellation rate:      30.1 %
  share of cancellations: 30.8 %

phone bookings to or from the airport
  share of bookings:      15.0 %
  cancellation rate:      0.8 %
  share of cancellations: 1.7 %

Per 1,000 bookings: flagged 256 | cancellations 74 | cancellations inside the flagged list 44


**Question 9.**

**Who gets flagged.** The model flagged 513 of the 2,000 test bookings. 17% of them actually cancelled, compared with 4% of the ones it didn't flag.

* **Channel:** 76% of flagged bookings came through the website or mobile site, against 29% of unflagged ones.
* **Airport:** only 3% of flagged bookings are airport runs, against 35% of the rest.
* **Trip type:** 93% of flagged bookings are regular point to point rides.
* **Time of day:** 44% of flagged pickups are in the evening (5pm to midnight), against 23% of unflagged ones.
* **Lead time:** bookings made one to seven days ahead cancelled least (3.8% on the test set). Same-day bookings cancelled at 7% to 8%, and ones made more than a week out at 11.8%.

**Putting numbers on it.** The profile shows which features travel together, but they overlap. Website bookings also skew toward evening rides, for example. To separate them, we fit a logistic regression on the training set as an explanation tool alongside the KNN model and read it through average marginal effects: the change in the chance of cancelling for a booking that's identical in every other way.

| Change (holding everything else equal) | Logit effect | KNN what-if |
|---|---|---|
| Mobile site instead of phone | +17.4 pts | +14.2 pts |
| Website instead of phone | +10.2 pts | +8.8 pts |
| Evening pickup instead of daytime | +3.1 pts | +5.0 pts (7pm vs 11am) |
| Airport trip instead of not | -7.3 pts | -7.0 pts |
| Booked over a week ahead instead of 1 to 7 days | +13.0 pts | -1.3 pts (10 days vs 3) |
| Booked under 3 hours ahead instead of 1 to 7 days | +8.4 pts | +0.3 pts (2 hours vs 3 days) |

The average cancellation rate is 7.4%, so these are big moves. All the logit effects above are significant at p < 0.001. On channel, airport and time of day, the two models agree closely, so those are solid findings. On lead time they don't. The raw rates and the logit both show the U shape, but the KNN model barely leans on it, so we'd treat lead time as a softer finding.

**Post-regression check.** Every VIF in the logistic regression is below 3.4 (highest: `point_to_point` at 3.31 and `package_hours` at 3.21). No coefficient is being distorted by overlap with the others, so the effects above can be read one at a time. The regression also ranks test bookings about as well as KNN does (test AUC 0.774 against 0.778).

**For the operations team, in plain English:**

The bookings most likely to fall through are ordinary city rides booked on the website or app, especially in the evening. A booking made on the app is about 14 to 17 points more likely to cancel than the same trip booked by phone, and a website booking about 9 to 10 points more likely. Phone bookings cancel around 4% of the time, so an app booking is several times riskier. Airport trips run the other way: about 7 points *less* likely to cancel than an otherwise identical ride.

What to do, with the numbers behind it:

1. **Confirm every evening website or app city ride before committing a car.** These are only 7.6% of bookings, but 30% of them cancel, and they account for 31% of all cancellations. Per 1,000 bookings, that's about 76 confirmation texts to reach roughly 23 of the 74 cancellations.
2. **If there's capacity for more, use the model's full flagged list.** Per 1,000 bookings it flags 256, and 44 of the 74 cancellations are in that list. That's a wider net for a bit more than three times the follow-up work.
3. **Push app bookings to confirm in the app.** Channel is the biggest single difference we found, so a "tap to confirm" message two hours before pickup is the cheapest place to start.
4. **Don't plan a car for every evening app booking until it's confirmed.** With 3 in 10 of those rides not happening, holding a driver for each one ties up capacity that could go to other customers.
5. **Skip phone bookings to or from the airport.** They're 15% of all bookings but cancel less than 1% of the time (1.7% of all cancellations). Any time spent checking on them is wasted.
6. **Send a reminder the day before for bookings made more than a week out.** About 5% of bookings fall here and around 9% cancel. It's worth a cheap reminder, though this pattern was weaker in the model than the others.

In [37]:
# How much of the final table rests on estimates rather than records?
print("Training bookings with an imputed pickup point:", (~has_pickup).sum(), "of", len(train))
print("Bookings with a lead time set to 0 or capped:",
      ((train['from_date'] - train['booking_created']).dt.total_seconds() / 3600 < 0).sum()
      + ((train['from_date'] - train['booking_created']).dt.total_seconds() / 3600 > lead_cap).sum())
print()

# The cancellation rate by pickup month, which the model never saw.
y_train.groupby(train['from_date'].dt.month).mean().round(3)

Training bookings with an imputed pickup point: 13 of 8000
Bookings with a lead time set to 0 or capped: 84



from_date
1     0.014
2     0.028
3     0.017
4     0.048
5     0.127
6     0.077
7     0.032
8     0.035
9     0.079
10    0.149
11    0.184
12    0.000
Name: Car_Cancellation, dtype: float64

**Question 10.**

What we'd tell the client about the limits of this analysis:

* **The pickup points we filled in barely matter.** Only 13 training bookings needed a pickup point estimated. The estimate isn't much better than guessing the typical pickup spot, but it's so few rows that it can't move the results.
* **Dropping the return time was a judgment call.** `to_date` wasn't recorded for most of the first half of the year, and a lot of the recorded values were wrong. Dropping it was the safe choice, but it means the model knows nothing about how long a trip was planned to last. If the company now records return times reliably, that could become a useful feature.
* **Some features depend on how things were recorded.** The airport flag matches one exact set of coordinates. If the booking system logs a new terminal or slightly different coordinates, the flag will quietly stop working. The common vehicle flag also assumes model 12 stays the main car in the fleet.
* **84 lead times were adjusted.** Bookings created after their pickup were set to 0, and very far-ahead ones were capped at 25 days. Those are reasonable fixes, but they're our choices, not facts from the data.
* **The biggest risk is time.** Cancellations were around 1% to 3% in January to March and 15% to 18% in October and November. The model never sees the date, and because the split was random, the test set is spread over the same months as the training set. So the test results show how well the model handles *this year's mix* of bookings, not how well it would do next month. If the cancellation rate keeps shifting, the 0.10 cutoff will flag too many or too few bookings.
* **The effects in Question 9 are associations, not proof of cause.** App customers may simply be people with looser plans. Moving them to the phone wouldn't necessarily make them cancel less. The numbers say who to watch, not what will change their behavior.
* **It's one city.** Everything here is Bangalore, so it says nothing about other markets.

What we'd want to check before trusting it on live bookings:

1. **Re-test it by time.** Train on the earlier months and test on the last two or three. That's the closest thing to how the model will actually be used, and it's the honest measure of how well it holds up.
2. **Run it quietly for a few weeks.** Score real bookings without acting on them, then check whether the flagged ones really cancel at around 17% and whether the model still catches about half or more of the cancellations.
3. **Confirm the data is still recorded the same way.** Make sure the airport coordinates, the vehicle codes and the booking channel flags mean what they meant in this file.
4. **Test the recommendations, not just the model.** For example, send confirmation texts to a random half of flagged bookings for a month and compare cancellations and wasted trips against the other half. That's the only way to know how much the actions actually save.
5. **Figure out why cancellations jumped late in the year.** If it was a pricing change, a new app release or a driver shortage, that's a bigger story than anything this model can see, and it might need its own fix.